In [13]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

## Load dataset

In [2]:
def calculate_macro_tpr_fpr(voting_cm):
    num_classes = voting_cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        TP = voting_cm[i, i]
        FN = np.sum(voting_cm[i, :]) - TP
        FP = np.sum(voting_cm[:, i]) - TP
        TN = np.sum(voting_cm) - (TP + FN + FP)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    macro_tpr = np.mean(tpr_list)
    macro_fpr = np.mean(fpr_list)

    return macro_tpr, macro_fpr

train = pd.read_csv('./dataset/CFM_augmented_9600.csv')
test = pd.read_csv('./test_shap_65.csv')

In [3]:
X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']
# label_encoder = LabelEncoder()
# y_train = label_encoder.fit_transform(y_train)
# y_test = label_encoder.transform(y_test)

In [4]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

## XGBoost

In [ ]:
num_classes = len(np.unique(y_train))
xgb_params = {
    'tree_method': 'hist',
    'device': 'cpu',
    'max_depth': 10,
    'n_estimators': 3000,
    'learning_rate': 0.65,
    'eval_metric': 'mlogloss',
    'objective': 'multi:softprob',
    'num_class': num_classes,
    'booster': 'gbtree',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1
}

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_start_time = time.time()
xgb_prediction = xgb_model.predict(X_test)
xgb_end_time = time.time()
xgb_time = xgb_end_time - xgb_start_time
print("XGBClassifier Finished")

xgb_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=xgb_prediction)
xgb_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=xgb_prediction)

print("XGBoost report:")
print("XGBoost Time:", xgb_time)
print("XGBoost Accuracy:", xgb_acc)
print("XGBoost Precision:", xgb_precision)
print("XGBoost F1:", xgb_f1)
print("XGBoost Recall:", xgb_recall)
print("XGBoost CM:\n", xgb_cm)
xgb_tpr, xgb_fpr = calculate_macro_tpr_fpr(xgb_cm)
print(f'XGBoost Macro-average TPR: {xgb_tpr}')
print(f'XGBoost Macro-average FPR: {xgb_fpr}')

XGBClassifier Starting
XGBClassifier Finished
XGBoost report:
XGBoost Time: 0.07005643844604492
XGBoost Accuracy: 0.8446745562130178
XGBoost Precision: 0.8372747299804518
XGBoost F1: 0.8378206966821753
XGBoost Recall: 0.8446745562130178
XGBoost CM:
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   1 168   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  38   1   1   0   0  56  50   8  14   0]
 [  0   0   0   0   0   0   0  42 127   0   0   0]
 [  0   0  10   0   0   0   1   7   0  87  64   0]
 [  0   0   0   0   0   0   0   0   0  77  92   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
XGBoost Macro-average TPR: 0.8446745562130178
XGBoost Macro-average FPR: 0.014120494889725659


## ExtraTree

In [9]:
et_params = {
    "n_estimators": 19,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
et_prediction = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")

et_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=et_prediction)
et_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=et_prediction)
et_fp = et_cm[0, 1]
print("ExtraTrees report:")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

ExtraTreesClassifier Starting
ExtraTreesClassifier Finished
ExtraTrees report:
ExtraTrees Time: 0.017966508865356445
ExtraTrees Accuracy: 0.8589743589743589
ExtraTrees Precision: 0.8574876221926865
ExtraTrees F1: 0.8551270484309854
ExtraTrees Recall: 0.858974358974359
ExtraTrees FP: 0
ExtraTrees CM:
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   1   0 168   0   0   0   0   0   0]
 [  0   0   0   1   0   0 168   0   0   0   0   0]
 [  0   1  25   0   0   0   0  79  35  12  17   0]
 [  0   0   0   0   0   0   0  36 133   0   0   0]
 [  0   0  10   0   0   0   0   6   0 107  46   0]
 [  0   0   0   0   0   0   0   0   0  96  73   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
ExtraTrees Macro-average TPR: 0.858974358974359
ExtraTrees Macro-average FPR: 0.01282

In [6]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 10, 30, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            19  0.858974  0.855127      0.858974  0.022044
1            21  0.858974  0.854954      0.858974  0.022404
2            12  0.857495  0.853441      0.857495  0.017099
3            17  0.857495  0.853392      0.857495  0.020937
4            22  0.857495  0.853042      0.857495  0.021764
5            18  0.857002  0.852952      0.857002  0.021712
6            15  0.857002  0.852481      0.857002  0.017784
7            23  0.856509  0.851793      0.856509  0.020489
8            11  0.856016  0.852120      0.856016  0.019530
9            20  0.855030  0.850358      0.855030  0.022551

===== Best ExtraTrees report =====
Best n_estimators: 19
ExtraTrees Time: 0.022043704986572266
ExtraTrees Accuracy: 0.8589743589743589
ExtraTrees Precision: 0.8574876221926865
ExtraTrees F1: 0.8551270484309854
ExtraTrees Recall:

## RandomForest

In [14]:
rf_params = {
    "n_estimators": 217,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
rf_prediction = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")

rf_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=rf_prediction)
rf_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=rf_prediction)
rf_fp = rf_cm[0, 1]
print("RandomForest report:")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest CM: \n", rf_cm)
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest report:
RandomForest Time: 0.09199380874633789
RandomForest Accuracy: 0.854043392504931
RandomForest Precision: 0.8504658866733564
RandomForest F1: 0.8482143723161543
RandomForest Recall: 0.854043392504931
RandomForest CM: 
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   0 169   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  37   1   1   0   4  70  35   7  13   0]
 [  0   0   0   0   0   0   0  35 134   0   0   0]
 [  0   0  13   0   0   0   0   4   0 101  51   0]
 [  0   0   0   0   0   0   0   0   0  94  75   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
RandomForest Macro-average TPR: 0.854043392504931
RandomForest Macro-average FPR: 0.0

In [11]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu bạn đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    """
    cm: confusion_matrix (C x C)
    Trả về (macro_TPR, macro_FPR)
    """
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 210, 230, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = sklearn.accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = sklearn.precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = sklearn.f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = sklearn.recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0           217  0.854043  0.848214      0.854043  0.103885
1           216  0.853550  0.847765      0.853550  0.102581
2           226  0.853550  0.847765      0.853550  0.103076
3           225  0.853057  0.847386      0.853057  0.115498
4           214  0.853057  0.847316      0.853057  0.106468
5           215  0.853057  0.847306      0.853057  0.105436
6           219  0.853057  0.847306      0.853057  0.101499
7           227  0.853057  0.847306      0.853057  0.245126
8           213  0.852564  0.846924      0.852564  0.103845
9           212  0.852564  0.846857      0.852564  0.105319

===== Best RandomForest report =====
Best n_estimators: 217
RandomForest Time: 0.10388517379760742
RandomForest Accuracy: 0.854043392504931
RandomForest Precision: 0.8504658866733564
RandomForest F1: 0.8482143723161543
Rando

In [23]:
joblib.dump(xgb_model, './models/xgb_model_cfm.pkl')
joblib.dump(et_model, './models/et_model_cfm.pkl')
joblib.dump(rf_model, './models/rf_model_cfm.pkl')

print("Models saved to ./models/")
print("  - xgb_model_cfm.pkl")
print("  - et_model_cfm.pkl")
print("  - rf_model_cfm.pkl")

Models saved to ./models/
  - xgb_model_cfm.pkl
  - et_model_cfm.pkl
  - rf_model_cfm.pkl
